# Explore silver Data

In [1]:
from processing.spark.utils.spark_session import create_spark_session

In [2]:
spark = create_spark_session("explore-silver")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 18:25:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Movies

In [3]:
movies_silver = spark.read.format("delta").load("s3a://silver/cleaned/movies")

26/04/07 17:33:27 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
movies_silver.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- genres_raw: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



In [5]:
movies_silver.limit(10).show(truncate = False)

26/04/07 17:33:37 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+----------------------+------------+------------------------+------------------------+--------+--------------------------+
|movie_id|title                 |release_year|genres_raw              |ingestion_timestamp     |source  |processed_timestamp       |
+--------+----------------------+------------+------------------------+------------------------+--------+--------------------------+
|4       |Waiting to Exhale     |1995        |Comedy|Drama|Romance    |2026-03-31 14:40:37.8339|postgres|2026-04-07 12:07:51.890363|
|7       |Sabrina               |1995        |Comedy|Romance          |2026-03-31 14:40:37.8339|postgres|2026-04-07 12:07:51.890363|
|8       |Tom and Huck          |1995        |Adventure|Children      |2026-03-31 14:40:37.8339|postgres|2026-04-07 12:07:51.890363|
|9       |Sudden Death          |1995        |Action                  |2026-03-31 14:40:37.8339|postgres|2026-04-07 12:07:51.890363|
|15      |Cutthroat Island      |1995        |Action|Adventure|Romanc

In [6]:
movies_silver.describe().show()

+-------+------------------+--------------------+------------------+------------------+--------+
|summary|          movie_id|               title|      release_year|        genres_raw|  source|
+-------+------------------+--------------------+------------------+------------------+--------+
|  count|            103993|              103993|            103993|            103993|  103993|
|   mean|168797.19387843413|            Infinity|1995.6441106612945|              null|    null|
| stddev| 81595.89924212045|                 NaN|25.883939251467872|              null|    null|
|    min|                 1|"BLOW THE NIGHT!"...|              1888|(no genres listed)|postgres|
|    max|            300373|            줄탁동시|              2024|           unknown|postgres|
+-------+------------------+--------------------+------------------+------------------+--------+



In [7]:
movies_genres = spark.read.format("delta").load("s3a://silver/enriched/movie_genres")

In [8]:
movies_genres.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- genre: string (nullable = true)



In [9]:
movies_genres.limit(10).show(truncate = False)

+--------+---------+
|movie_id|genre    |
+--------+---------+
|1       |adventure|
|1       |animation|
|1       |children |
|1       |comedy   |
|1       |fantasy  |
|2       |adventure|
|2       |children |
|2       |fantasy  |
|3       |comedy   |
|3       |romance  |
+--------+---------+



In [10]:
movies_genres.describe().show()

+-------+------------------+-------+
|summary|          movie_id|  genre|
+-------+------------------+-------+
|  count|            172883| 172883|
|   mean|161196.72173088157|   null|
| stddev| 85497.06928051771|   null|
|    min|                 1| action|
|    max|            300373|western|
+-------+------------------+-------+



___

## Belief Data

In [11]:
belief_silver = spark.read.parquet("s3a://silver/cleaned/belief")

In [12]:
belief_silver.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- is_seen: integer (nullable = true)
 |-- watch_date: date (nullable = true)
 |-- user_rating: double (nullable = true)
 |-- predicted_rating: double (nullable = true)
 |-- user_certainty: double (nullable = true)
 |-- system_rating: double (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- movie_idx: integer (nullable = true)
 |-- source_type: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



In [13]:
belief_silver.limit(10).show(truncate = False)

+-------+--------+-------+----------+-----------+----------------+--------------+----------------+-------------------+---------+-----------+--------------------------+-------------+--------------------------+
|user_id|movie_id|is_seen|watch_date|user_rating|predicted_rating|user_certainty|system_rating   |event_timestamp    |movie_idx|source_type|ingestion_timestamp       |source_system|processed_timestamp       |
+-------+--------+-------+----------+-----------+----------------+--------------+----------------+-------------------+---------+-----------+--------------------------+-------------+--------------------------+
|1892   |1147    |null   |null      |null       |null            |null          |4.33950812174257|2023-03-19 07:40:07|0        |2          |2026-03-31 14:40:52.370411|postgres     |2026-04-07 11:45:10.342698|
|1892   |1147    |null   |null      |null       |null            |null          |4.33598895434672|2023-07-30 04:57:44|4        |2          |2026-03-31 14:40:52.3704

In [14]:
belief_silver.describe().show()

+-------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------+
|summary|          user_id|          movie_id|            is_seen|       user_rating|  predicted_rating|    user_certainty|     system_rating|         movie_idx|       source_type|source_system|
+-------+-----------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------+
|  count|          3004084|           3004084|              37027|             37027|             37027|             37027|           3004084|           3004084|           3004084|      3004084|
|   mean|339709.6425702477|118793.36241563152|0.23131768709320225|0.9055013908769276|2.1290949847408647|2.3593593863937126| 3.814156253959741|  6.72236961416525|1.7499956725577581|         null|
| stddev|84074.4166460542

In [15]:
belief_features = spark.read.parquet("s3a://silver/enriched/belief_features")

In [16]:
belief_features.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- is_seen: integer (nullable = true)
 |-- has_watched: boolean (nullable = true)
 |-- user_rating: double (nullable = true)
 |-- system_rating: double (nullable = true)
 |-- rating_diff: double (nullable = true)
 |-- user_certainty: double (nullable = true)
 |-- is_high_confidence: boolean (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



In [17]:
belief_features.limit(10).show(truncate = False)

+-------+--------+-------+-----------+-----------+----------------+-----------+--------------+------------------+-------------------+----------+
|user_id|movie_id|is_seen|has_watched|user_rating|system_rating   |rating_diff|user_certainty|is_high_confidence|event_timestamp    |event_date|
+-------+--------+-------+-----------+-----------+----------------+-----------+--------------+------------------+-------------------+----------+
|399680 |21      |null   |null       |null       |3.5642070099969 |null       |null          |null              |2023-07-23 13:53:45|2023-07-23|
|399712 |172     |null   |null       |null       |2.76433906391306|null       |null          |null              |2023-07-23 09:37:25|2023-07-23|
|399915 |172     |null   |null       |null       |2.76433906391306|null       |null          |null              |2023-07-23 12:20:42|2023-07-23|
|399642 |208     |null   |null       |null       |2.87953090184147|null       |null          |null              |2023-07-23 06:24:

In [18]:
belief_features.describe().show()

+-------+-----------------+------------------+-------------------+------------------+------------------+-------------------+------------------+
|summary|          user_id|          movie_id|            is_seen|       user_rating|     system_rating|        rating_diff|    user_certainty|
+-------+-----------------+------------------+-------------------+------------------+------------------+-------------------+------------------+
|  count|          3004084|           3004084|              37027|             37027|           3004084|              37027|             37027|
|   mean|339709.6425702477|118793.36241563152|0.23131768709320225|0.9055013908769276| 3.814156253959734|-2.9044668620196075|2.3593593863937126|
| stddev|84074.41664605416|117444.91944475859|0.42168070506858096|1.7002598084287979|0.6360039553543076| 1.7665756026699437|1.6792135409303965|
|    min|             1892|                 1|                  0|               0.0|               0.0|               -5.0|            

---

## Movie Elicitation

In [19]:
elicitation_silver = spark.read.parquet("s3a://silver/cleaned/movie_elicitation_set")

In [20]:
elicitation_silver.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- month_idx: integer (nullable = true)
 |-- source_type: integer (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



In [21]:
elicitation_silver.limit(10).show(truncate = False)

+--------+---------+-----------+-------------------+--------------------------+-------------+--------------------------+
|movie_id|month_idx|source_type|event_timestamp    |ingestion_timestamp       |source_system|processed_timestamp       |
+--------+---------+-----------+-------------------+--------------------------+-------------+--------------------------+
|1       |0        |1          |2023-02-27 19:30:03|2026-03-31 14:40:37.407635|postgres     |2026-04-07 12:34:31.501541|
|1       |1        |1          |2023-04-01 00:01:47|2026-03-31 14:40:37.407635|postgres     |2026-04-07 12:34:31.501541|
|1       |2        |1          |2023-05-01 00:02:01|2026-03-31 14:40:37.407635|postgres     |2026-04-07 12:34:31.501541|
|1       |3        |1          |2023-06-01 00:01:40|2026-03-31 14:40:37.407635|postgres     |2026-04-07 12:34:31.501541|
|1       |4        |1          |2023-07-01 00:01:56|2026-03-31 14:40:37.407635|postgres     |2026-04-07 12:34:31.501541|
|1       |5        |1          |

In [22]:
elicitation_silver.describe().show()

+-------+------------------+------------------+------------------+-------------+
|summary|          movie_id|         month_idx|       source_type|source_system|
+-------+------------------+------------------+------------------+-------------+
|  count|             84058|             84058|             84058|        84058|
|   mean|124368.98924552095| 2.509945513811892|1.8462609150824432|         null|
| stddev|111243.58484842019|3.0017823590299106|1.1935393107451158|         null|
|    min|                 1|                 0|                 1|     postgres|
|    max|            299713|                14|                 5|     postgres|
+-------+------------------+------------------+------------------+-------------+



In [23]:
elicitation_features = spark.read.parquet("s3a://silver/enriched/elicitation_features")

In [24]:
elicitation_features.printSchema()

root
 |-- movie_id: integer (nullable = true)
 |-- month_idx: integer (nullable = true)
 |-- source_type: integer (nullable = true)
 |-- source_category: string (nullable = true)
 |-- month_group: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- is_recent: boolean (nullable = true)
 |-- event_date: date (nullable = true)



In [25]:
elicitation_features.limit(10).show(truncate = False)

+--------+---------+-----------+---------------+-----------+-------------------+---------+----------+
|movie_id|month_idx|source_type|source_category|month_group|event_timestamp    |is_recent|event_date|
+--------+---------+-----------+---------------+-----------+-------------------+---------+----------+
|1032    |1        |1          |popular        |early_stage|2023-04-01 00:01:47|false    |2023-04-01|
|1301    |1        |1          |popular        |early_stage|2023-04-01 00:01:47|false    |2023-04-01|
|1465    |1        |1          |popular        |early_stage|2023-04-01 00:01:47|false    |2023-04-01|
|1527    |1        |1          |popular        |early_stage|2023-04-01 00:01:47|false    |2023-04-01|
|1937    |1        |1          |popular        |early_stage|2023-04-01 00:01:47|false    |2023-04-01|
|2262    |1        |1          |popular        |early_stage|2023-04-01 00:01:47|false    |2023-04-01|
|2671    |1        |1          |popular        |early_stage|2023-04-01 00:01:47|fa

In [26]:
elicitation_features.describe().show()

+-------+------------------+-----------------+------------------+---------------+-----------+
|summary|          movie_id|        month_idx|       source_type|source_category|month_group|
+-------+------------------+-----------------+------------------+---------------+-----------+
|  count|             84058|            84058|             84058|          84058|      84058|
|   mean|124368.98924552095|2.509945513811892|1.8462609150824432|           null|       null|
| stddev|111243.58484841938|3.001782359029911|1.1935393107451246|           null|       null|
|    min|                 1|                0|                 1|        popular|early_stage|
|    max|            299713|               14|                 5|     well_rated|  mid_stage|
+-------+------------------+-----------------+------------------+---------------+-----------+



---

## Ratings

In [27]:
ratings_silver = spark.read.parquet("s3a://silver/cleaned/ratings")

In [28]:
ratings_silver.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_topic: string (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



In [29]:
ratings_silver.limit(10).show(truncate = False)

+-------+--------+------+-------------------+--------------------------+-------------+--------------------------------+--------------------------+
|user_id|movie_id|rating|event_timestamp    |ingestion_timestamp       |source_system|source_topic                    |processed_timestamp       |
+-------+--------+------+-------------------+--------------------------+-------------+--------------------------------+--------------------------+
|1892   |235     |3.5   |2004-10-31 00:41:16|2026-03-30 18:19:59.541793|kafka        |csv-ratings_for_additional_users|2026-04-07 14:16:07.504423|
|1892   |357     |3.0   |2007-10-17 01:42:18|2026-03-30 18:19:59.541793|kafka        |csv-ratings_for_additional_users|2026-04-07 14:16:07.504423|
|1892   |422     |3.0   |1997-09-28 11:53:17|2026-03-30 18:19:59.541793|kafka        |csv-ratings_for_additional_users|2026-04-07 14:16:07.504423|
|1892   |506     |1.0   |2006-01-13 22:58:08|2026-03-30 18:19:59.541793|kafka        |csv-ratings_for_additional_users

In [30]:
ratings_silver.describe().show()

+-------+------------------+-----------------+------------------+-------------+--------------------+
|summary|           user_id|         movie_id|            rating|source_system|        source_topic|
+-------+------------------+-----------------+------------------+-------------+--------------------+
|  count|          12357052|         12357052|          10808014|     12357052|            12357052|
|   mean| 290150.0261207932|77332.93193586949| 3.444699831069797|         null|                null|
| stddev|101073.37599035275|  82085.650423774|1.0324681699192395|         null|                null|
|    min|              1892|                1|               0.5|        kafka|csv-ratings_for_a...|
|    max|            410572|           300349|               5.0|        kafka|csv-ratings_for_a...|
+-------+------------------+-----------------+------------------+-------------+--------------------+



In [3]:
ratings_features = spark.read.parquet("s3a://silver/enriched/rating_features")

26/04/07 18:03:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/04/07 18:06:40 WARN SharedInMemoryCache: Evicting cached table partition metadata from memory due to size constraints (spark.sql.hive.filesourcePartitionFileCacheSize = 262144000 bytes). This may impact query planning performance.


In [4]:
ratings_features.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- rating_category: string (nullable = true)
 |-- is_positive: boolean (nullable = true)
 |-- rating_bucket: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- recency_days: integer (nullable = true)
 |-- event_date: date (nullable = true)



In [5]:
ratings_features.limit(10).show(truncate = False)

+-------+--------+------+---------------+-----------+-------------+-------------------+------------+----------+
|user_id|movie_id|rating|rating_category|is_positive|rating_bucket|event_timestamp    |recency_days|event_date|
+-------+--------+------+---------------+-----------+-------------+-------------------+------------+----------+
|406188 |68954   |5.0   |high           |true       |top          |2024-01-07 02:58:39|821         |2024-01-07|
|406173 |509     |3.5   |medium         |false      |mid          |2024-01-07 00:59:21|821         |2024-01-07|
|406173 |1201    |null  |null           |null       |null         |2024-01-07 00:53:51|821         |2024-01-07|
|406173 |5234    |null  |null           |null       |null         |2024-01-07 12:13:39|821         |2024-01-07|
|406173 |31528   |null  |null           |null       |null         |2024-01-07 12:02:10|821         |2024-01-07|
|406173 |58120   |null  |null           |null       |null         |2024-01-07 11:36:58|821         |2024

In [ ]:
ratings_features.describe().show()

---

## Recommendation History

In [3]:
recommendation_silver = spark.read.parquet("s3a://silver/cleaned/recommendation_history")

26/04/07 18:25:32 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
recommendation_silver.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- predicted_rating: double (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_topic: string (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



In [5]:
recommendation_silver.limit(10).show(truncate = False)

+-------+--------+----------------+-------------------+--------------------------+-------------+-------------------------------+--------------------------+
|user_id|movie_id|predicted_rating|event_timestamp    |ingestion_timestamp       |source_system|source_topic                   |processed_timestamp       |
+-------+--------+----------------+-------------------+--------------------------+-------------+-------------------------------+--------------------------+
|42170  |76093   |4.36934309167091|2023-04-08 19:34:40|2026-03-30 18:26:09.172485|kafka        |csv-user_recommendation_history|2026-04-07 16:57:44.462347|
|42170  |76093   |4.36925032094962|2023-05-28 21:40:53|2026-03-30 18:26:17.499336|kafka        |csv-user_recommendation_history|2026-04-07 16:57:44.462347|
|42170  |97921   |4.83508485143862|2023-05-26 18:15:09|2026-03-30 18:26:16.603392|kafka        |csv-user_recommendation_history|2026-04-07 16:57:44.462347|
|43715  |94959   |3.97297542549029|2023-03-31 21:01:42|2026-03-3

In [6]:
recommendation_silver.describe().show(truncate = False)

26/04/07 18:25:37 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+------------------+-------------+-------------------------------+
|summary|user_id           |movie_id         |predicted_rating  |source_system|source_topic                   |
+-------+------------------+-----------------+------------------+-------------+-------------------------------+
|count  |1285352           |1285352          |1285352           |1285352      |1285352                        |
|mean   |342883.53297773685|78510.9044339605 |4.429982335875917 |null         |null                           |
|stddev |79879.42550841259 |93794.25769296093|0.4177536908741118|null         |null                           |
|min    |42170             |1                |0.444623023621201 |kafka        |csv-user_recommendation_history|
|max    |410438            |299065           |5.0               |kafka        |csv-user_recommendation_history|
+-------+------------------+-----------------+------------------+-------------+-------------------------

In [7]:
recommendation_features = spark.read.parquet("s3a://silver/enriched/recommendation_features")

In [8]:
recommendation_features.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- predicted_rating: double (nullable = true)
 |-- predicted_rating_category: string (nullable = true)
 |-- is_high_score: boolean (nullable = true)
 |-- predicted_rating_bucket: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)



In [9]:
recommendation_features.limit(10).show(truncate = False)

+-------+--------+----------------+-------------------------+-------------+-----------------------+-------------------+----------+
|user_id|movie_id|predicted_rating|predicted_rating_category|is_high_score|predicted_rating_bucket|event_timestamp    |event_date|
+-------+--------+----------------+-------------------------+-------------+-----------------------+-------------------+----------+
|406699 |253498  |5.0             |high                     |true         |top                    |2024-01-21 15:28:16|2024-01-21|
|404902 |165549  |4.97300579975879|high                     |true         |top                    |2024-01-21 15:54:16|2024-01-21|
|377140 |1196    |4.01747658137875|high                     |true         |high                   |2024-01-21 16:42:44|2024-01-21|
|404737 |527     |4.99069990707734|high                     |true         |top                    |2024-01-21 17:48:57|2024-01-21|
|406781 |318     |4.28999865866906|high                     |true         |high    

In [10]:
recommendation_features.describe().show(truncate = False)

+-------+------------------+-----------------+-------------------+-------------------------+-----------------------+
|summary|user_id           |movie_id         |predicted_rating   |predicted_rating_category|predicted_rating_bucket|
+-------+------------------+-----------------+-------------------+-------------------------+-----------------------+
|count  |1285352           |1285352          |1285352            |1285352                  |1285352                |
|mean   |342883.53297773685|78510.9044339605 |4.429982335875906  |null                     |null                   |
|stddev |79879.42550841291 |93794.25769296093|0.41775369087411035|null                     |null                   |
|min    |42170             |1                |0.444623023621201  |high                     |high                   |
|max    |410438            |299065           |5.0                |medium                   |very_low               |
+-------+------------------+-----------------+------------------

---

In [11]:
spark.stop()